In [1]:
# Variogram model fitting and LOOCV — FI
# Fits spherical, exponential, and Gaussian models (isotropic and anisotropic)
# and ranks them by leave-one-out cross-validation. The final model adopted for
# mapping is the anisotropic spherical (see the paper and the mapping cell below).

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import curve_fit
from pykrige.ok import OrdinaryKriging
from numpy.linalg import LinAlgError

file_path = "Kriging_GN.xlsx"
target = "FI"

df = pd.read_excel(file_path, sheet_name="Sheet1").dropna(
    subset=["Easting", "Northing", target]).copy()
X = df["Easting"].to_numpy(float)
Y = df["Northing"].to_numpy(float)
V = df[target].to_numpy(float)

ANG_DEG = 40.0
RATIO_MINOR_MAJOR = 0.253
MIN_PAIRS = 100
nlags = 15

def experimental_variogram(x, y, z, maxlag, nlags):
    coords = np.c_[x, y]
    D = squareform(pdist(coords))
    iu = np.triu_indices(len(z), 1)
    h = D[iu]
    gamma_pairs = (0.5 * (z[:, None] - z[None, :]) ** 2)[iu]
    edges = np.linspace(0, maxlag, nlags + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    gamma, npairs = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (h > a) & (h <= b)
        npairs.append(m.sum())
        gamma.append(gamma_pairs[m].mean() if m.sum() > 0 else np.nan)
    return centers, np.array(gamma), np.array(npairs)

def rotate_and_scale(x, y, angle_deg, ratio):
    xc, yc = x - x.mean(), y - y.mean()
    theta = np.deg2rad(angle_deg)
    xr = xc * np.cos(theta) + yc * np.sin(theta)
    yr = -xc * np.sin(theta) + yc * np.cos(theta)
    return xr, yr / ratio

def spherical(h, nugget, psill, a):
    h = np.asarray(h)
    return nugget + psill * np.where(h < a, 1.5 * (h / a) - 0.5 * (h / a) ** 3, 1.0)

def exponential(h, nugget, psill, a):
    h = np.asarray(h)
    return nugget + psill * (1.0 - np.exp(-h / a))

def gaussian(h, nugget, psill, a):
    h = np.asarray(h)
    return nugget + psill * (1.0 - np.exp(-(h / a) ** 2))

models = {"spherical": spherical, "exponential": exponential, "gaussian": gaussian}

def fit_variogram_model(model_name, centers, gamma, npairs, maxlag):
    used = np.isfinite(gamma) & (npairs >= MIN_PAIRS)
    c, g, n = centers[used], gamma[used], npairs[used]
    if len(g) < 4:
        return None
    nug0 = max(0.0, np.nanmin(g))
    sill0 = max(np.nanpercentile(g, 80), nug0 + 1e-8)
    p0 = [nug0, max(sill0 - nug0, 1e-8), 0.4 * maxlag]
    f = models[model_name]
    popt, _ = curve_fit(f, c, g, p0=p0,
                        bounds=([0.0, 1e-10, 5000.0], [np.inf, np.inf, maxlag]),
                        sigma=1.0 / np.sqrt(n), maxfev=50000)
    nugget, psill, rng = popt
    return {"nugget": float(nugget), "psill": float(psill), "range": float(rng)}

def loocv_ok(coords, values, family, params, anisotropic=False):
    errors = []
    for i in range(len(values)):
        keep = np.ones(len(values), dtype=bool); keep[i] = False
        try:
            OK = OrdinaryKriging(
                coords[keep, 0], coords[keep, 1], values[keep],
                variogram_model=family,
                variogram_parameters={"psill": params["psill"], "range": params["range"], "nugget": params["nugget"]},
                anisotropy_angle=ANG_DEG if anisotropic else 0.0,
                anisotropy_scaling=RATIO_MINOR_MAJOR if anisotropic else 1.0,
                enable_plotting=False, verbose=False, pseudo_inv=True)
            z_pred, _ = OK.execute("points", np.array([coords[i, 0]]), np.array([coords[i, 1]]))
            errors.append(values[i] - float(np.asarray(z_pred).ravel()[0]))
        except (LinAlgError, ValueError) as e:
            print(f"LOOCV skipped point {i}: {e}")
            continue
    errors = np.array(errors)
    return {"RMSE": float(np.sqrt(np.mean(errors ** 2))),
            "MAE": float(np.mean(np.abs(errors))),
            "ME": float(np.mean(errors)),
            "N_used": int(len(errors))}

coords = np.c_[X, Y].astype(float)
maxlag_iso = 0.5 * pdist(coords).max()
X_a, Y_a = rotate_and_scale(X, Y, ANG_DEG, RATIO_MINOR_MAJOR)
maxlag_ani = 0.5 * pdist(np.c_[X_a, Y_a]).max()

cent_iso, gam_iso, pairs_iso = experimental_variogram(X, Y, V, maxlag_iso, nlags)
cent_ani, gam_ani, pairs_ani = experimental_variogram(X_a, Y_a, V, maxlag_ani, nlags)

rows = []
for name in models:
    fi = fit_variogram_model(name, cent_iso, gam_iso, pairs_iso, maxlag_iso)
    fa = fit_variogram_model(name, cent_ani, gam_ani, pairs_ani, maxlag_ani)
    if fi:
        r = loocv_ok(coords, V, name, fi, anisotropic=False)
        rows.append({"Family": name, "Case": "Isotropic", **fi, **r})
    if fa:
        r = loocv_ok(coords, V, name, fa, anisotropic=True)
        rows.append({"Family": name, "Case": "Anisotropic", **fa, **r})

results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
print("Variogram models ranked by LOOCV RMSE (FI):")
print(results.to_string(index=False))
print("\nFinal model adopted for mapping: anisotropic spherical")

Variogram models ranked by LOOCV RMSE (FI):
     Family        Case   nugget    psill         range     RMSE      MAE        ME  N_used
   gaussian   Isotropic 0.003070 0.000193 106341.270491 0.055487 0.044171 -0.000020     318
  spherical Anisotropic 0.002728 0.000456  84545.010674 0.055699 0.044342 -0.000012     318
exponential   Isotropic 0.002958 0.000225  33422.250479 0.055883 0.044544 -0.000007     318
   gaussian Anisotropic 0.002810 0.000375  42821.114655 0.055891 0.044401 -0.000024     318
  spherical   Isotropic 0.002881 0.000273  41293.224729 0.055915 0.044296 -0.000015     318
exponential Anisotropic 0.002526 0.000657  22474.714150 0.056575 0.045060  0.000026     318

Final model adopted for mapping: anisotropic spherical
